# Kaggle ASTGCN 与 baseline 性能对比

本 notebook 面向 Kaggle/Ubuntu 环境，用于自动定位项目代码和 PEMS04 数据文件，复用 `src/astgcn` 包与 `scripts/compare_baselines.py`，比较 `HA`、`SVR`、`LSTM`、`GRU`、`ASTGCN` 的 MAE、RMSE、MAPE，并生成指标表和预测曲线。

默认使用 `quick` 模式做快速连通性检查；正式实验时将 `RUN_MODE` 改为 `full`。

In [ ]:
!git clone https://github.com/Tuzfucius/ASTGCN-learning

from pathlib import Path
import os
import subprocess
import sys


def find_project_root():
    candidates = [Path.cwd(), Path('/kaggle/working/ASTGCN'), Path('/kaggle/working')]
    for start in candidates:
        if not start.exists():
            continue
        for path in [start, *start.parents]:
            if (path / 'pyproject.toml').exists() and (path / 'src' / 'astgcn').exists():
                return path
        for child in start.glob('**/pyproject.toml'):
            root = child.parent
            if (root / 'src' / 'astgcn').exists():
                return root
    raise FileNotFoundError('未找到 ASTGCN 项目根目录')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'], check=True)


In [ ]:
import json
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display


## 运行配置

- `quick`：用于 Kaggle 快速调试，只跑少量 batch，并限制 SVR 样本数。
- `full`：使用更多 batch 和配置中的 epoch 数，适合正式对比实验。

In [ ]:
RUN_MODE = 'quick'  # 正式实验可改为 'full'

with open(PROJECT_ROOT / 'configs' / 'pems04.yaml', 'r', encoding='utf-8') as file:
    cfg = yaml.safe_load(file)


def find_existing_file(names):
    search_roots = [
        PROJECT_ROOT / 'data' / 'raw' / 'PEMS04',
        PROJECT_ROOT / 'data' / 'PEMS04',
        Path('/kaggle/input'),
        Path('/kaggle/working'),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
        for name in names:
            matches = list(root.glob(f'**/{name}'))
            if matches:
                return matches[0]
    raise FileNotFoundError(f'未找到数据文件: {names}')


data_path = find_existing_file(['pems04.npz', 'PEMS04.npz'])
distance_path = find_existing_file(['distance.csv'])
print('data_path =', data_path)
print('distance_path =', distance_path)

run_cfg = deepcopy(cfg)
run_cfg['dataset']['data_path'] = str(data_path)
run_cfg['dataset']['distance_path'] = str(distance_path)
run_cfg['train']['device'] = 'auto'

if RUN_MODE == 'quick':
    epochs = 1
    max_batches = 1
    svr_samples = 128
else:
    epochs = int(run_cfg['train']['epochs'])
    max_batches = 20
    svr_samples = 2048

CONFIG_FOR_RUN = PROJECT_ROOT / 'outputs' / 'comparison' / 'kaggle_pems04.yaml'
CONFIG_FOR_RUN.parent.mkdir(parents=True, exist_ok=True)
with open(CONFIG_FOR_RUN, 'w', encoding='utf-8') as file:
    yaml.safe_dump(run_cfg, file, allow_unicode=True, sort_keys=False)
print('CONFIG_FOR_RUN =', CONFIG_FOR_RUN)
print('epochs =', epochs, 'max_batches =', max_batches, 'svr_samples =', svr_samples)


## 训练与性能对比

运行统一对比脚本，训练或评估 `HA`、`SVR`、`LSTM`、`GRU`、`ASTGCN`，并将结果保存到 `outputs/comparison/`。

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'compare_baselines.py'),
    '--config', str(CONFIG_FOR_RUN),
    '--epochs', str(epochs),
    '--max-batches', str(max_batches),
    '--svr-samples', str(svr_samples),
    '--device', run_cfg['train']['device'],
    '--output-dir', str(PROJECT_ROOT / 'outputs' / 'comparison'),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 指标结果

MAE 和 RMSE 越低越好；MAPE 对接近 0 的真实值更敏感，因此需要结合 MAE/RMSE 一起判断。

In [ ]:
comparison_dir = PROJECT_ROOT / 'outputs' / 'comparison'
metrics_path = comparison_dir / 'baseline_metrics.csv'
metrics = pd.read_csv(metrics_path)
display(metrics)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
colors = ['#4c78a8', '#f58518', '#54a24b', '#e45756', '#72b7b2']
for axis, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    axis.bar(metrics['model'], metrics[metric], color=colors[:len(metrics)])
    axis.set_title(metric)
    axis.set_ylabel(metric)
    axis.grid(axis='y', alpha=0.25)
    axis.tick_params(axis='x', rotation=30)
plt.show()


## 单节点预测曲线

选择一个样本和一个节点，查看各模型在未来 `T_p` 个时间步上的预测曲线。

In [ ]:
pred_file = comparison_dir / 'baseline_predictions.npz'
arrays = np.load(pred_file)
target = arrays['target']
model_names = [name for name in arrays.files if name != 'target']

sample_id = 0
node_id = 0
steps = np.arange(1, target.shape[-1] + 1)

plt.figure(figsize=(11, 5))
plt.plot(steps, target[sample_id, node_id], marker='o', linewidth=2.4, color='#111111', label='Target')
for name in model_names:
    plt.plot(steps, arrays[name][sample_id, node_id], marker='o', linewidth=1.5, label=name)
plt.title(f'Baseline prediction comparison: sample={sample_id}, node={node_id}')
plt.xlabel('Prediction step')
plt.ylabel('Traffic flow')
plt.grid(alpha=0.25)
plt.legend(ncol=2)
plt.show()


## 输出文件

本次运行会生成：

- `outputs/comparison/baseline_metrics.csv`
- `outputs/comparison/baseline_metrics.json`
- `outputs/comparison/metrics_bar.png`
- `outputs/comparison/sample_prediction.png`
- `outputs/comparison/baseline_predictions.npz`